# NL_SQL autotune - Colab runner (duplicate of the Kaggle S3 run)

Second shot at the same QLoRA training, so a lost Kaggle session does not cost another day.
Same `train_qlora.py`, same dataset - both are pulled from the Kaggle dataset
`liovinajo/nlsql-autotune-data`, which is the single source of truth for this track.

**Colab differs from Kaggle in one way that matters:** `/content` is wiped on every
disconnect, and the free tier disconnects often. So checkpoints live on Google Drive and the
run resumes from them. Re-running this notebook after a drop continues where it stopped.

Steps: Runtime -> Change runtime type -> **T4 GPU**, then Run all. Two prompts appear:
Drive authorisation, and an upload box for `kaggle.json` (Kaggle -> Settings -> Create New
API Token). Nothing else is interactive.

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
KAGGLE_DATASET = "liovinajo/nlsql-autotune-data"

DRIVE_DIR = "/content/drive/MyDrive/nlsql_autotune"  # checkpoints + adapter survive here
DATA_DIR = "/content/nlsql_data"  # re-downloaded per session, never on Drive
OUT = DRIVE_DIR + "/qlora_out"

EPOCHS = 1.0
MAX_SEQ = 4096
SMOKE_STEPS = 5  # >0: prove the whole path on N steps first (0 = straight into the real run)

In [ ]:
import subprocess
import sys
from pathlib import Path

from google.colab import drive, files

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout, flush=True)

drive.mount("/content/drive")
Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)

# kaggle.json: reuse the copy on Drive if a previous session left one, else ask once.
cred = Path("/root/.kaggle/kaggle.json")
cred.parent.mkdir(parents=True, exist_ok=True)
drive_cred = Path(DRIVE_DIR) / "kaggle.json"
if drive_cred.exists():
    cred.write_bytes(drive_cred.read_bytes())
    print("kaggle.json taken from Drive", flush=True)
else:
    print("upload kaggle.json (Kaggle -> Settings -> Create New API Token)", flush=True)
    for name, blob in files.upload().items():
        if name.endswith(".json"):
            cred.write_bytes(blob)
            drive_cred.write_bytes(blob)  # so the next session after a drop is unattended
cred.chmod(0o600)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
subprocess.run(
    ["kaggle", "datasets", "download", "-d", KAGGLE_DATASET, "-p", DATA_DIR, "--unzip"],
    check=True,
)
print(sorted(p.name for p in Path(DATA_DIR).iterdir()), flush=True)

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unsloth"], check=True)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "datasets>=3.0",
        "trl>=0.12",
        "peft>=0.13",
        "bitsandbytes>=0.44",
        "accelerate>=1.0",
    ],
    check=True,
)

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

data_dir = Path(DATA_DIR)
script = data_dir / "train_qlora.py"
if not script.exists():
    raise RuntimeError("train_qlora.py missing from the downloaded dataset")


def train_cmd(out, extra):
    return [
        sys.executable,
        str(script),
        "--model",
        BASE_MODEL,
        "--train",
        str(data_dir / "train.jsonl"),
        "--val",
        str(data_dir / "val.jsonl"),
        "--out",
        out,
        "--max-seq-len",
        str(MAX_SEQ),
        "--batch-size",
        "1",
        "--grad-accum",
        "16",
        "--no-merge",
        *extra,
    ]


if SMOKE_STEPS and not Path(OUT, "checkpoints").exists():
    # Only on a fresh start - after a disconnect the path is already proven and the
    # session time is better spent resuming. Smoke writes to /content, not Drive.
    smoke_out = "/content/smoke_out"
    smoke = train_cmd(smoke_out, ["--epochs", "1", "--max-steps", str(SMOKE_STEPS)])
    print(" ".join(smoke), flush=True)
    subprocess.run(smoke, check=True)
    if not Path(smoke_out, "adapter", "adapter_config.json").exists():
        raise RuntimeError("smoke run finished but saved no adapter")
    shutil.rmtree(smoke_out)
    print("SMOKE OK -- starting the real run", flush=True)

cmd = train_cmd(OUT, ["--epochs", str(EPOCHS), "--resume"])
print(" ".join(cmd), flush=True)
subprocess.run(cmd, check=True)
shutil.make_archive(DRIVE_DIR + "/adapter", "zip", OUT, "adapter")
print("DONE: " + DRIVE_DIR + "/adapter.zip", flush=True)